# Universal Parallel Scaling Laws

This notebook analyzes parallel performance using **Guided Scheduling**.

We compare theoretical models (Amdahl's and Gustafson's Laws) with real-world GPU+MPI data.

In [ ]:
import subprocess
import re
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
procs = [2, 4, 8, 16, 24]
times = []
N = 2048 # Matrix size

for p in procs:
    cmd = [
        "mpirun", "--oversubscribe",
        "-n", str(p),
        "./scripts/parallel_conv",
        str(N)
    ]

    print(f"Benchmarking {p} processors...")
    res = subprocess.run(cmd, capture_output=True, text=True)
    
    match = re.search(r"Time taken: ([0-9.]+)", res.stdout)
    if match:
        times.append(float(match.group(1)))
    else:
        print("Error in execution")

## Scaling Analysis

In [ ]:
p_arr = np.array(procs)
t_arr = np.array(times)
speedup = t_arr[0] * (procs[0]/p_arr) # Relative to first multi-core run
actual_speedup = t_arr[0] / t_arr

s = 0.05 # estimated serial fraction
amdahl = 1 / (s + (1-s)/p_arr)
gustafson = p_arr - s*(p_arr-1)

plt.figure(figsize=(10, 6))
plt.plot(p_arr, amdahl, '--', label="Amdahl's Law (Theoretical)")
plt.plot(p_arr, actual_speedup, 'o-', label="Actual Speedup (Guided Scheduling)")

plt.legend()
plt.xlabel("Processors")
plt.ylabel("Speedup")
plt.title("Scaling Performance Analysis")
plt.grid(True)
plt.show()